# Módulo 05: Análisis exploratorio de datos (EDA)

> El EDA no es "hacer muchas gráficas": es **hacerle preguntas a los datos** y dejar que las respuestas guíen el modelado.

**Objetivos**
1. Entender la variable objetivo (`SalePrice`).
2. Explorar variables numéricas y categóricas, solas (**univariado**) y frente al target (**bivariado**).
3. Detectar *outliers* y decidir qué hacer con ellos.
4. **Validar las hipótesis** que planteamos en el módulo 03.
5. Salir con una lista de **variables candidatas** para el modelo.

> Basado en la sección *Análisis exploratorio* de `sesion_3_preparacion_de_datos_desarrollo.ipynb` (profesor Oscar N. Gómez).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from intro_analitica.datos import cargar_house_prices_limpio
from intro_analitica.eda import correlaciones_con_target, resumen_outliers

sns.set_theme(style="whitegrid")
df = cargar_house_prices_limpio()  # salida del módulo 04 (si no existe, se genera)
print(df.shape)

Trabajaremos con un subconjunto de variables (la selección del profesor, más las que creamos en el módulo 04):

In [ ]:
numericas = ["LotArea", "OverallQual", "OverallCond", "YearBuilt", "YearRemodAdd", "GrLivArea",
             "TotalBsmtSF", "FullBath", "BedroomAbvGr", "GarageCars", "HouseAge", "TotalSF", "TotalBath"]
categoricas = ["Neighborhood", "BldgType", "HouseStyle", "KitchenQual", "SaleCondition"]
target = "SalePrice"

house = df[numericas + categoricas + [target]]
house.describe().T.round(1)

---
## 1. La variable objetivo

Siempre se empieza por aquí: **¿qué estamos prediciendo y cómo se distribuye?**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
sns.histplot(house[target], bins=50, kde=True, ax=axes[0])
axes[0].set(title=f"SalePrice (asimetría = {house[target].skew():.2f})")
sns.violinplot(y=house[target], ax=axes[1])
axes[1].set(title="Violín de SalePrice")
sns.histplot(np.log1p(house[target]), bins=50, kde=True, ax=axes[2], color="seagreen")
axes[2].set(title=f"log(SalePrice) (asimetría = {np.log1p(house[target]).skew():.2f})")
plt.tight_layout()
plt.show()

**Lectura:**
- El precio tiene **cola a la derecha** (asimetría ≈ 1.9): muchas casas "normales" y pocas muy caras.
- Con **logaritmo** queda casi simétrico (≈ 0.1). Lo usaremos como experimento en el módulo 06: muchos modelos lineales funcionan mejor con un target simétrico.

---
## 2. Análisis univariado

### 2.1 Numéricas

In [ ]:
house[numericas].hist(bins=30, figsize=(14, 9), layout=(4, 4), color="steelblue")
plt.suptitle("Distribución de las variables numéricas", y=1.01)
plt.tight_layout()
plt.show()

**Qué buscar en estos histogramas:**

| Patrón | Ejemplo | Implicación |
|---|---|---|
| Cola larga | `LotArea`, `GrLivArea` | Posibles *outliers*; considerar logaritmo |
| Discreta / pocos valores | `OverallQual`, `FullBath`, `GarageCars` | Es ordinal o un conteo |
| Pico en 0 | `TotalBsmtSF` | Casas **sin** sótano (lo vimos en el módulo 04) |
| Bimodal | `YearBuilt` | Dos "olas" de construcción |

### 2.2 Categóricas

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, col in zip(axes, ["BldgType", "HouseStyle", "KitchenQual", "SaleCondition"]):
    house[col].value_counts().plot(kind="bar", ax=ax, color="darkorange")
    ax.set(title=col, xlabel="")
plt.tight_layout()
plt.show()

Hay categorías **dominantes** (`BldgType = 1Fam`, `SaleCondition = Normal`) y otras **raras** (pocas casas).
Las categorías raras son un riesgo: el modelo aprende poco de ellas, y en el one-hot generan columnas casi vacías.

---
## 3. Análisis bivariado: cada variable frente al precio

### 3.1 Correlaciones

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(house[numericas + [target]].corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Matriz de correlación (Pearson)")
plt.show()

In [ ]:
correlaciones_con_target(house, target).to_frame("r_con_SalePrice")

**Dos lecturas de la matriz:**
1. **Frente al target:** `OverallQual` (0.79), `TotalSF` (0.78) y `GrLivArea` (0.71) son las variables más relacionadas con el precio.
2. **Entre variables (multicolinealidad):** `TotalSF` contiene a `TotalBsmtSF`; `HouseAge` y `YearBuilt` son casi la misma información (r ≈ -1). Usar ambas en una regresión lineal **no agrega información** y vuelve inestables los coeficientes.

> Pearson solo mide relaciones **lineales**. Una r baja no significa "no hay relación".

### 3.2 Numéricas vs. precio

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), ["OverallQual", "TotalSF", "GrLivArea", "GarageCars", "HouseAge", "LotArea"]):
    sns.regplot(data=house, x=col, y=target, ax=ax, scatter_kws={"alpha": 0.25, "s": 10}, line_kws={"color": "red"})
    ax.set(title=f"{col} (r = {house[col].corr(house[target]):.2f})")
plt.tight_layout()
plt.show()

### 3.3 Categóricas vs. precio

In [ ]:
orden = house.groupby("Neighborhood")[target].median().sort_values().index
plt.figure(figsize=(12, 7))
sns.boxplot(data=house, y="Neighborhood", x=target, order=orden, color="lightsteelblue")
plt.title("Precio por barrio (ordenado por mediana)")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["KitchenQual", "BldgType", "HouseStyle"]):
    orden_col = house.groupby(col)[target].median().sort_values().index
    sns.violinplot(data=house, x=col, y=target, order=orden_col, ax=ax, linewidth=1)
    ax.set(title=f"{col} vs. SalePrice")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

`KitchenQual` muestra una escalera muy clara: `Fa < TA < Gd < Ex`. Es **ordinal**, y el orden se refleja en el precio.

---
## 4. *Outliers*: detectar ≠ eliminar

### 4.1 Regla del IQR

In [ ]:
resumen_outliers(house).head(8)

¿`TotalBath` tiene un 27 % de *outliers*? No: es una variable **discreta** (1, 1.5, 2, 2.5…) con el IQR muy estrecho.
**La regla del IQR es una alerta, no una sentencia.** Siempre hay que mirar el gráfico y el contexto.

### 4.2 Los *outliers* famosos de House Prices

In [ ]:
grandes = house[house["GrLivArea"] > 4000]
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.scatterplot(data=house, x="GrLivArea", y=target, alpha=0.3, ax=ax)
sns.scatterplot(data=grandes, x="GrLivArea", y=target, color="red", s=80, ax=ax, label="GrLivArea > 4000")
ax.set(title="Las casas gigantes")
plt.show()

df.loc[grandes.index, ["GrLivArea", "SalePrice", "Neighborhood", "SaleCondition", "OverallQual"]]

Dos casas **enormes** (>4.500 pies²) y de calidad 10 se vendieron **baratas** (~USD 170.000), las dos en `Edwards` y con `SaleCondition = Partial` (casas nuevas, vendidas antes de terminarse).

| Opción | Pros | Contras |
|---|---|---|
| Eliminarlas | Una regresión lineal no se "tuerce" por 2 puntos | Si llega una casa así, el modelo no la habrá visto |
| Dejarlas | Representan la realidad | Pueden dominar el error (RMSE) |
| Usar un modelo robusto | Los árboles son menos sensibles | — |

**Decisión para el curso:** las **dejamos en el dataset** y lo anotamos como riesgo. En el módulo 06 compararemos modelos con y sin ellas.
El propio autor del dataset (De Cock, 2011) recomienda quitar las casas de más de 4.000 pies² en ejercicios de clase.

---
## 5. Validación de hipótesis (del módulo 03)

### H1: la calidad general es el factor que más explica el precio

In [ ]:
house.groupby("OverallQual")[target].agg(["size", "median"]).astype(int)

**Confirmada.** Tiene la mayor correlación (0.79) y una relación creciente y casi exponencial: cada punto de calidad sube el precio mediano entre un 20 % y un 45 %.

### H2: a mayor área, mayor precio (con *outliers*)
**Confirmada** (r = 0.71 para `GrLivArea`, 0.78 para `TotalSF`), con las 2 excepciones de `Edwards` vistas arriba.

### H3: el barrio influye aun comparando casas de la misma calidad

In [ ]:
calidad_7 = house[house["OverallQual"] == 7]
por_barrio = calidad_7.groupby("Neighborhood")[target].agg(["size", "median"])
por_barrio[por_barrio["size"] >= 15].sort_values("median").astype(int)

**Confirmada.** Entre casas de calidad 7 (y con al menos 15 ventas por barrio), el precio mediano va de ~USD 156.000 (`OldTown`) a ~USD 228.000 (`SawyerW`): un **46 % más** solo por la ubicación.

### H4: las casas remodeladas valen más que otras de la misma edad

In [ ]:
house_h4 = house.assign(
    rango_edad=pd.cut(house["HouseAge"], bins=[-1, 10, 30, 50, 80], labels=["0-10", "11-30", "31-50", "51-80"]),
    remodelada=df["Remodeled"].map({0: "No", 1: "Sí"}),
)
tabla_h4 = house_h4.groupby(["rango_edad", "remodelada"], observed=True)[target].median().unstack()
tabla_h4["diferencia_%"] = ((tabla_h4["Sí"] / tabla_h4["No"] - 1) * 100).round(1)
tabla_h4

**Confirmada (con matices).** En todos los rangos de edad, las remodeladas tienen una mediana mayor (entre un 5 % y un 12 %).

Fíjate que la correlación simple entre `Remodeled` y `SalePrice` era **≈ 0** (lo vimos en el módulo 04). La relación aparece solo cuando **comparamos casas de la misma edad**: las remodeladas suelen ser más viejas, y eso "esconde" el efecto. Esto se conoce como **variable de confusión**.

---
## 6. Conclusiones del EDA (el entregable)

| # | Hallazgo | Implicación para el modelo |
|---|---|---|
| 1 | `SalePrice` es asimétrico; con log queda simétrico | Probar el target en escala log |
| 2 | `OverallQual`, `TotalSF`, `GrLivArea` y `GarageCars` son las variables más fuertes | Variables imprescindibles |
| 3 | El barrio importa aun controlando la calidad | Incluir `Neighborhood` (one-hot) |
| 4 | `KitchenQual` es ordinal y escalonada | Ordinal encoding u one-hot |
| 5 | `HouseAge` ≈ `YearBuilt` (r ≈ -1) | Usar solo una en los modelos lineales |
| 6 | 2 casas gigantes y baratas (`Partial`, `Edwards`) | Riesgo: evaluar con y sin ellas |
| 7 | El efecto de remodelar aparece solo comparando por edad | Los modelos multivariados lo captan; ojo con conclusiones univariadas |

En el **módulo 06** usaremos estos hallazgos para construir el *baseline* y los primeros modelos.

---
## Ejercicios guiados (20 min)

1. **Precio por m²:** crea `precio_m2 = SalePrice / (GrLivArea × 0.0929)`. ¿Cuál es el barrio más caro por m²? ¿Es el mismo que el más caro en precio total?
2. **Estacionalidad del precio:** el volumen de ventas sube en verano (módulo 02). ¿Y el **precio mediano**? Grafica la mediana de `SalePrice` por `MoSold`.
3. **Outliers con k = 3:** usa `outliers_iqr(serie, k=3)` sobre `LotArea`. ¿Cuántos quedan? Muéstralos en un scatter frente al precio.
4. **Una hipótesis nueva:** plantea y valida una (p.ej. "las casas con chimenea valen más", usando `Fireplaces`).

Soluciones en `ejercicios/soluciones/solucion_05.py`.

In [ ]:
# Tu código aquí